# 01 - Análisis Exploratorio de IOCs

**Objetivo:** Explorar los datasets crudos de ThreatFox, MalwareBazaar y URLhaus
para entender su estructura, calidad y utilidad para construir features.

**Datasets:**
- `threatfox.csv`
- `malwarebazaar.csv`
- `urlhaus.csv`

In [1]:
# Cell 2 - Configuración
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Pandas:", pd.__version__)
print("DATA_DIR:", DATA_DIR.resolve())

Pandas: 3.0.6
DATA_DIR: /mnt/e/ConquerBlocks/Proyecto JTL/threat_intelligence/ai/data/raw


In [2]:
COLS = [
    "first_seen_utc", "sha256_hash", "md5_hash", "sha1_hash", "reporter",
    "file_name", "file_type_guess", "mime_type", "signature", "clamav",
    "vtpercent", "imphash", "ssdeep", "tlsh",
]

def load_malwarebazaar(path: str | Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\r\n")
            if not line or line.startswith("#"):
                continue
            parts = line.split('", "')
            # Limpieza de bordes
            parts[0] = parts[0].lstrip('" ').rstrip('"')
            parts[-1] = parts[-1].rstrip('"').strip()
            if len(parts) == len(COLS):
                rows.append(parts)
            # else: línea malformada → podrías loguearla
    return pd.DataFrame(rows, columns=COLS)



def load_default(path):
    return pd.read_csv(path)   # si es CSV estándar



In [3]:
# Cell 3 - Carga
datasets = {
    "threatfox": load_default(DATA_DIR / "threatfox.csv"),
    "malwarebazaar": load_malwarebazaar(DATA_DIR / "malwarebazaar.csv"),
    "urlhaus": load_default(DATA_DIR / "urlhaus.csv"),
}

def resumen(df, nombre):
    print(f"\n{'='*60}\n{nombre}\n{'='*60}")
    print("Shape:", df.shape)
    print("Columnas:", df.columns.tolist())
    print("\nNulos (top 10):")
    print(df.isnull().sum().sort_values(ascending=False).head(10))
    print("\nDuplicados:", df.duplicated().sum())

for nombre, df in datasets.items():
    resumen(df, nombre)


threatfox
Shape: (11941, 17)
Columnas: ['id', 'ioc', 'threat_type', 'threat_type_desc', 'ioc_type', 'ioc_type_desc', 'malware', 'malware_printable', 'malware_alias', 'malware_malpedia', 'confidence_level', 'is_compromised', 'first_seen', 'last_seen', 'reference', 'reporter', 'tags']

Nulos (top 10):
last_seen            9730
reference            9158
malware_alias        7573
tags                  986
id                      0
ioc_type                0
ioc                     0
threat_type_desc        0
threat_type             0
malware_printable       0
dtype: int64

Duplicados: 0

malwarebazaar
Shape: (1383, 14)
Columnas: ['first_seen_utc', 'sha256_hash', 'md5_hash', 'sha1_hash', 'reporter', 'file_name', 'file_type_guess', 'mime_type', 'signature', 'clamav', 'vtpercent', 'imphash', 'ssdeep', 'tlsh']

Nulos (top 10):
first_seen_utc     0
sha256_hash        0
md5_hash           0
sha1_hash          0
reporter           0
file_name          0
file_type_guess    0
mime_type          0
s

In [4]:
# Cell 4 - Análisis específico ThreatFox
tf = datasets["threatfox"]
print(tf["ioc_type"].value_counts(normalize=True).mul(100).round(2))
print(tf["malware_printable"].value_counts().head(10))
print(tf["confidence_level"].describe())

ioc_type
sha256_hash    50.72
domain         28.83
ip:port        10.08
url             7.19
md5_hash        1.71
sha1_hash       1.47
Name: proportion, dtype: float64
malware_printable
Mirai              3373
AsyncRAT           2008
Unknown Loader     1342
IClickFix          1022
ClearFake           831
Unknown malware     498
AdaptixC2           280
Vidar               256
Remus               240
AMOS                220
Name: count, dtype: int64
count    11941.000000
mean        84.606817
std         18.603077
min         50.000000
25%         75.000000
50%         95.000000
75%        100.000000
max        100.000000
Name: confidence_level, dtype: float64


In [5]:
# Cell 5 - Análisis temporal
tf["first_seen_utc"] = pd.to_datetime(tf["first_seen_utc"], errors="coerce")
tf["last_seen_utc"]  = pd.to_datetime(tf["last_seen_utc"], errors="coerce")
print("Rango first_seen:", tf["first_seen_utc"].min(), "→", tf["first_seen_utc"].max())
print("IOCs activos últimos 30d:",
      (tf["last_seen_utc"] > pd.Timestamp.now() - pd.Timedelta(days=30)).sum())

KeyError: 'first_seen_utc'

In [ ]:
# Cell 6 - Tags
tags = tf["tags"].dropna().astype(str).str.split(",").explode().str.strip()
print(tags.value_counts().head(15))

In [ ]:
# Cell 7 - Visualizaciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
tf["ioc_type"].value_counts().plot(kind="bar", ax=axes[0], title="Tipos de IOC")
tf["malware_printable"].value_counts().head(10).plot(kind="barh", ax=axes[1], title="Top 10 familias")
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig(REPORTS_DIR / "01_ioc_distribucion.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close(fig)

## Hallazgos

- **Total de IOCs:** X registros
- **Tipos predominantes:** domain (~%), ip:port (~%), url (~%), md5/sha256 (~%)
- **Fuentes más activas:** reporter1, reporter2, ...
- **Familias más reportadas:** ...

### Campos útiles para features
- `ioc_type`, `malware_printable`, `confidence_level`, `first_seen_utc`,
  `last_seen_utc`, `reporter`, `tags`, TLD del dominio, longitud del IOC.

### Limitaciones detectadas
- **Sesgo:** sobrerrepresentación de ciertas familias (ej. Cobalt Strike).
- **Datos expirados:** registros con `last_seen_utc` muy antiguo.
- **Nulos:** columnas como `malware_printable` o `tags` con muchos vacíos.
- **Duplicados:** IOCs repetidos entre fuentes → riesgo de leakage.